# RCM check 4 - Trends in the stabilised 2050 runs, and their effect on the hosing response

**Ad hoc test, outside the chain**: it reads the same files as notebook 3 (cache of notebook 1 for EC-Earth3 and
RegCM5 0.25 deg, monthly files of RegCM5 0.75 deg prepared by ECOADAPT in `ECOADAPT/AMOC/outputs/preproc/075/<run>/mon`,
read only) and writes only figures / tables in `outputs/` next to this notebook.

**Question.** The runs are stabilisation runs (fixed greenhouse-gas concentrations): their climate should have no
trend. The ECOADAPT drift check (`drift_check_025_20261029_b`, annual land mean of the 2 m temperature TSA, EUR-075)
found no significant trend in three runs, but -0.23 K per decade (p = 0.01) in stab-2050, the no-hosing run of the
2050 pair; the hosing run has none. The land surface of the RCM was hardly spun up (the 2050 runs start in December
2124; the deep soil water fills for 16-18 years). If one run of the pair drifts and the other does not, the hosing
effect (hosing - no hosing, `YEARS`) contains the difference of the two drifts. (1) Is the trend in the RCM only (spin-up
artefact) or also in EC-Earth3 (climate still adjusting)? (2) In which seasons and regions? (3) Does it change the
hosing effect of notebooks 1-3, in particular the stronger autumn cooling of RegCM5 0.75 deg (-1.62 K against -0.64 K)?

**Method.** As notebook 3: both RCMs on the EC-Earth3 grid (0.25 deg: mean of the cells whose centre falls in a GCM
cell; 0.75 deg: nearest cell; relaxation zones left out), cells covered by both RCMs in the European LISFLOOD domain,
the 8 PRUDENCE regions; tas = (tasmax + tasmin) / 2, close to TSA. For each model, run, variable, season and region: one
value per year, linear trend per decade over 2125-2149 and over `YEARS`, p-value with the effective sample size of the
lag-1 autocorrelated residuals (Santer et al. 2000); the same for the yearly difference hosing - no hosing. Hosing
effect three ways: (a) mean over `YEARS`, as notebooks 1-3; (b) mean over the last decade `LATE`; (c) difference of
the two linear fits (over `YEARS`) at the last year. If the drift is a spin-up artefact, the end of the run is the most
spun-up state: (c) removes the part of the drift still present in (a). The three agree when the drift does not matter.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, glob, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from scipy import sparse, stats
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
ECO = Path(os.environ.get("ECOADAPT_ROOT", Path(cfg["data_root"]).parent / "ECOADAPT" / "AMOC"))   # read only
R075_RUNS = {"nohos": "stabilization-ssp585-2050", "hos": "stabilization-ssp585-2050-hosing"}
RCM_TAGS = {"nohos": "2050_Base", "hos": "2050_Hose"}                     # cache of notebook 1 (RegCM5 0.25)
GCM_RUNS = {"nohos": "gcm_2050_nohosing", "hos": "gcm_2050_hosing"}
FILE_VARS = ["tasmax", "tasmin", "pr"]
VARS = ["tas", "tasmax", "tasmin", "pr"]      # tas = (tasmax + tasmin) / 2
DATA_YEARS = (2125, 2149)
YEARS = tuple(int(y) for y in os.environ.get("SIGNAL_YEARS", "2130,2149").split(","))
LATE = (YEARS[1] - 9, YEARS[1])
PERIODS = {f"{DATA_YEARS[0]}-{DATA_YEARS[1]}": DATA_YEARS, f"{YEARS[0]}-{YEARS[1]}": YEARS}
SPONGE = {"r025": 11 + 2, "r075": 11 + 2}     # cells left out along each RCM edge (relaxation zone + 2)
MIN_COVER = 0.5
P_SIG = 0.05
SEASONS = {"DJF": (12, 1, 2), "MAM": (3, 4, 5), "JJA": (6, 7, 8), "SON": (9, 10, 11), "year": tuple(range(1, 13))}
PRUDENCE = {"BI": ("British Isles", (50, 59), (-10, 2)), "IP": ("Iberian Peninsula", (36, 44), (-10, 3)),
            "FR": ("France", (44, 50), (-5, 5)), "ME": ("Mid-Europe", (48, 55), (2, 16)),
            "SC": ("Scandinavia", (55, 70), (5, 30)), "AL": ("Alps", (44, 48), (5, 15)),
            "MD": ("Mediterranean", (36, 44), (3, 25)), "EA": ("Eastern Europe", (44, 55), (16, 30))}
MODELS = {"gcm": "EC-Earth3", "r075": "RegCM5 0.75", "r025": "RegCM5 0.25"}
RUNS = {"nohos": "no hosing", "hos": "hosing", "diff": "hosing - no hosing"}

CACHE = bu.out_dir(cfg, "rcm_check", "cache")
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
TAG = f"{YEARS[0]}-{YEARS[1]}"
print(f"years {TAG}, last decade {LATE[0]}-{LATE[1]} | cache of notebook 1: {CACHE} | ECOADAPT: {ECO}")

## 1. Monthly means: EC-Earth3 and RegCM5 0.25 deg from the cache of notebook 1, RegCM5 0.75 deg from ECOADAPT

In [ ]:
DAYS = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], float)


def cached(path):
    assert path.exists(), f"missing {path.name}: run 1_hosing_signal_rcm_vs_gcm.ipynb first"
    return xr.open_dataarray(path).load()


def latlon2d(ds):
    for la, lo in (("lat", "lon"), ("xlat", "xlon"), ("latitude", "longitude")):
        if la in ds.variables and lo in ds.variables:
            a, o = ds[la].values.astype(float), ds[lo].values.astype(float)
            if a.ndim == 1:
                o, a = np.meshgrid(o, a)
            return a, ((o + 180.0) % 360.0) - 180.0
    raise KeyError(f"no lat / lon (variables: {list(ds.variables)})")


def months_of(ds):
    # (year, month) of each step (files opened with use_cftime): middle of the time bounds if present
    bn = [b for b in (ds["time"].attrs.get("bounds"), "time_bnds", "time_bounds") if b and b in ds]
    mid = [b0 + (b1 - b0) / 2 for b0, b1 in ds[bn[0]].values] if bn else list(ds["time"].values)
    return np.array([m.year for m in mid]), np.array([m.month for m in mid])


def to_units(a, units, var):
    u = str(units).lower().replace(" ", "")
    if var == "pr":
        return a * 86400.0 if u in ("kgm-2s-1", "kgm**-2s**-1", "kg/m2/s", "mm/s", "kgm^-2s^-1") else a
    return a - 273.15 if u in ("k", "kelvin") else a


def r075_monthly(run, var):
    fs = sorted(glob.glob(str(ECO / "outputs" / "preproc" / "075" / R075_RUNS[run] / "mon" / f"{var}_EUR-075_*_{R075_RUNS[run]}_*_mon_*.nc")))
    assert len(fs) >= 1, f"no 0.75 deg monthly file for {run} {var}"
    ds = xr.open_mfdataset(fs, combine="by_coords", use_cftime=True) if len(fs) > 1 else xr.open_dataset(fs[0], use_cftime=True)
    yr, mo = months_of(ds)
    keep = (yr >= DATA_YEARS[0]) & (yr <= DATA_YEARS[1])
    a = to_units(ds[var].squeeze(drop=True).values[keep].astype(float), ds[var].attrs.get("units", ""), var)
    lat, lon = latlon2d(ds)
    t = pd.DatetimeIndex([pd.Timestamp(int(y), int(m), 1) for y, m in zip(yr[keep], mo[keep])])
    assert t.size == 12 * (DATA_YEARS[1] - DATA_YEARS[0] + 1) and t.is_monotonic_increasing, f"{run} {var}: {t.size} months"
    return xr.DataArray(a, dims=("time", "y", "x"), coords={"time": t, "lat": (("y", "x"), lat), "lon": (("y", "x"), lon)}, name=var)


GCM = {(run, v): cached(CACHE / f"ecearth3_{GCM_RUNS[run]}_{v}_monthly_{DATA_YEARS[0]}-{DATA_YEARS[1]}.nc") for run in GCM_RUNS for v in FILE_VARS}
NAT = {("r025", run, v): cached(CACHE / f"rcm025_{RCM_TAGS[run]}_{v}_monthly_native_{DATA_YEARS[0]}-{DATA_YEARS[1]}.nc")
       for run in RCM_TAGS for v in FILE_VARS}
for run in R075_RUNS:
    for v in FILE_VARS:
        NAT["r075", run, v] = r075_monthly(run, v)
G0 = GCM["nohos", "tasmax"]
GLAT, GLON = G0.lat.values, G0.lon.values
print({k: f"{d.sizes['time']} months" for k, d in NAT.items() if k[2] == "tasmax"})

## 2. Aggregation onto the EC-Earth3 grid, cells covered by both RCMs, PRUDENCE regions (as notebook 3)

In [ ]:
def cell_area_km2(lat, lon):
    R = 6371.0
    la, lo = np.deg2rad(lat), np.deg2rad(lon)
    d = lambda a1, o1, a2, o2: 2 * R * np.arcsin(np.sqrt(np.sin((a2 - a1) / 2) ** 2
                                                       + np.cos(a1) * np.cos(a2) * np.sin((o2 - o1) / 2) ** 2))
    dx = d(la[:, :-1], lo[:, :-1], la[:, 1:], lo[:, 1:]); dx = np.c_[dx, dx[:, -1:]]
    dy = d(la[:-1], lo[:-1], la[1:], lo[1:]); dy = np.r_[dy, dy[-1:]]
    return dx * dy


ye, xe = bu._edges(GLAT), bu._edges(GLON)
NG = GLAT.size * GLON.size
g_area = (np.diff(np.sin(np.deg2rad(np.clip(ye, -90, 90))))[:, None] * np.deg2rad(np.diff(xe))[None, :]) * 6371.0 ** 2


def aggregator(da, sponge):
    # RCM grid -> EC-Earth3 grid: mean of the cells whose centre falls in a GCM cell; share of each GCM cell covered
    lat, lon = da["lat"].values.astype(float), ((da["lon"].values.astype(float) + 180.0) % 360.0) - 180.0
    ny, nx = lat.shape
    inner = np.zeros((ny, nx), bool); inner[sponge:ny - sponge, sponge:nx - sponge] = True
    iy, ix = np.searchsorted(ye, lat) - 1, np.searchsorted(xe, lon) - 1
    ok = inner & (iy >= 0) & (iy < GLAT.size) & (ix >= 0) & (ix < GLON.size)
    col = np.flatnonzero(ok); row = (iy * GLON.size + ix).ravel()[col]
    ones = sparse.csr_matrix((np.ones(col.size), (row, col)), shape=(NG, ny * nx))
    cover = (np.bincount(row, weights=cell_area_km2(lat, lon).ravel()[col], minlength=NG) / g_area.ravel()).reshape(GLAT.size, GLON.size)

    def to_gcm(a):
        flat = np.asarray(a, float).reshape(-1, ny * nx)
        fin = np.isfinite(flat)
        num = ones @ np.where(fin, flat, 0.0).T
        den = ones @ fin.T.astype(float)
        return np.where(den > 0, num / np.where(den > 0, den, 1), np.nan).T.reshape(flat.shape[0], GLAT.size, GLON.size)
    return to_gcm, cover


def nearest(da, sponge, max_deg=0.6):
    # 0.75 deg RCM (about as coarse as EC-Earth3): value of the nearest cell outside the relaxation zone within max_deg
    # of the GCM cell centre (the cell-centre mean would leave one GCM cell in two empty)
    from scipy.spatial import cKDTree
    lat, lon = da["lat"].values.astype(float), ((da["lon"].values.astype(float) + 180.0) % 360.0) - 180.0
    ny, nx = lat.shape
    inner = np.zeros((ny, nx), bool); inner[sponge:ny - sponge, sponge:nx - sponge] = True
    src = np.flatnonzero(inner.ravel())
    xyz = lambda la, lo: np.c_[np.cos(np.deg2rad(la)) * np.cos(np.deg2rad(lo)), np.cos(np.deg2rad(la)) * np.sin(np.deg2rad(lo)), np.sin(np.deg2rad(la))]
    gla, glo = np.meshgrid(GLAT, GLON, indexing="ij")
    dist, k = cKDTree(xyz(lat.ravel()[src], lon.ravel()[src])).query(xyz(gla.ravel(), glo.ravel()))
    ok = np.rad2deg(2 * np.arcsin(np.minimum(dist / 2, 1))) <= max_deg
    take = src[k]

    def to_gcm(a):
        flat = np.asarray(a, float).reshape(-1, ny * nx)[:, take]
        flat[:, ~ok] = np.nan
        return flat.reshape(-1, GLAT.size, GLON.size)
    return to_gcm, ok.reshape(GLAT.size, GLON.size).astype(float)


AGG = {"r025": aggregator(NAT["r025", "nohos", "tasmax"], SPONGE["r025"]), "r075": nearest(NAT["r075", "nohos", "tasmax"], SPONGE["r075"])}
DATA = {("gcm", run, v): GCM[run, v].assign_coords(lat=GLAT, lon=GLON) for run in GCM_RUNS for v in FILE_VARS}
for (m, run, v), da in NAT.items():
    DATA[m, run, v] = xr.DataArray(AGG[m][0](da.values), dims=("time", "lat", "lon"), coords={"time": da.time, "lat": GLAT, "lon": GLON})
for m in MODELS:
    for run in ("nohos", "hos"):
        DATA[m, run, "tas"] = 0.5 * (DATA[m, run, "tasmax"] + DATA[m, run, "tasmin"])

DOM = (bu.load_domain_mask(cfg, "ecearth3", like=G0.isel(time=0)).values
       & (AGG["r025"][1] >= MIN_COVER) & (AGG["r075"][1] >= MIN_COVER))
LAT2 = np.broadcast_to(GLAT[:, None], DOM.shape); LON2 = np.broadcast_to(GLON[None, :], DOM.shape)
WREG = {}
for r, (name, la, lo) in PRUDENCE.items():
    msk = DOM & (LAT2 >= la[0]) & (LAT2 <= la[1]) & (LON2 >= lo[0]) & (LON2 <= lo[1])
    w = np.where(msk, np.cos(np.deg2rad(LAT2)), 0.0).ravel()
    WREG[r] = w / w.sum()
w = np.where(DOM, np.cos(np.deg2rad(LAT2)), 0.0).ravel(); WREG["DOMAIN"] = w / w.sum()
REGIONS = ["DOMAIN"] + list(PRUDENCE)
print(f"GCM cells covered by both RCMs in the domain: {DOM.sum()}")

## 3. Yearly series per region, and their linear trends

pr: trends of the runs in % of their mean per decade; trend of the difference in percentage points per decade.

In [ ]:
def regional_series(da, months):
    # one value per year (DJF: year of the January), years with all months only, then the weighted mean of each region
    t = pd.DatetimeIndex(da["time"].values)
    sy = t.year + ((len(months) == 3 and 12 in months) & (t.month == 12)).astype(int)
    v, yrs, out = da.values, [], []
    for y in range(DATA_YEARS[0], DATA_YEARS[1] + 1):
        k = np.flatnonzero(np.isin(t.month, months) & (sy == y))
        if len(k) == len(months):
            wt = DAYS[t.month[k] - 1]
            out.append(np.tensordot(wt / wt.sum(), v[k], axes=1)); yrs.append(y)
    x = np.stack(out).reshape(len(yrs), -1)
    fin = np.isfinite(x); x0 = np.where(fin, x, 0.0)
    return np.array(yrs), np.stack([(x0 @ WREG[r]) / (fin @ WREG[r]) for r in REGIONS], axis=1)


def trend(yrs, x):
    # OLS slope per decade and its p-value, effective sample size n (1 - r1) / (1 + r1) from the lag-1
    # autocorrelation r1 of the residuals (Santer et al. 2000)
    b, a = np.polyfit(yrs, x, 1)
    res = x - (a + b * yrs)
    n = len(x)
    r1 = max(np.corrcoef(res[:-1], res[1:])[0, 1], 0.0) if np.std(res) > 0 else 0.0
    neff = max(n * (1 - r1) / (1 + r1), 3.0)
    se = np.sqrt(np.sum(res ** 2) / (neff - 2) / np.sum((yrs - yrs.mean()) ** 2))
    p = float(2 * stats.t.sf(abs(b) / se, neff - 2)) if se > 0 else 0.0
    return 10 * b, 10 * se, p, neff


def change(h, n, var):
    return 100.0 * (h / n - 1.0) if var == "pr" else h - n


SER = {}
for var in VARS:
    for s, months in SEASONS.items():
        for m in MODELS:
            yrs, xn = regional_series(DATA[m, "nohos", var], months)
            _, xh = regional_series(DATA[m, "hos", var], months)
            SER[var, s, m] = (yrs, xn, xh)

rows = []
for (var, s, m), (yrs, xn, xh) in SER.items():
    for run, x in (("nohos", xn), ("hos", xh), ("diff", change(xh, xn, var))):
        for pname, (p0, p1) in PERIODS.items():
            k = (yrs >= p0) & (yrs <= p1)
            for i, r in enumerate(REGIONS):
                sl, se, p, neff = trend(yrs[k].astype(float), x[k, i])
                if var == "pr" and run != "diff":
                    mu = x[k, i].mean(); sl, se = 100 * sl / mu, 100 * se / mu
                rows.append(dict(var=var, season=s, region=r, model=m, run=run, period=pname, slope_per_decade=sl, se=se,
                                 p=p, n_eff=neff, unit=("% / decade" if var == "pr" and run != "diff" else
                                                        "points / decade" if var == "pr" else "K / decade")))
TR = pd.DataFrame(rows)
TR.to_csv(OUT / f"trends_{DATA_YEARS[0]}-{DATA_YEARS[1]}.csv", index=False)

star = lambda p: "*" if p < P_SIG else " "
for var in VARS:
    t = TR[(TR["var"] == var) & (TR["region"] == "DOMAIN")]
    a, b = list(PERIODS)
    piv = {}
    for (m, run, s), g in t.groupby(["model", "run", "season"]):
        x, y = g[g["period"] == a].iloc[0], g[g["period"] == b].iloc[0]
        piv[(MODELS[m], RUNS[run]), s] = f"{x.slope_per_decade:+.2f}{star(x.p)} | {y.slope_per_decade:+.2f}{star(y.p)}"
    tab = pd.Series(piv).unstack()[list(SEASONS)]
    tab = tab.loc[[(MODELS[m], RUNS[r]) for m in MODELS for r in RUNS]]
    print(f"\n{var}: domain-mean trend per decade ({t['unit'].iloc[0]} for the runs), {a} | {b}, * p < {P_SIG}")
    display(tab)

## 4. Where is a drift significant? Regions and seasons, and whether EC-Earth3 drifts too

A trend of the RCM that EC-Earth3 does not have (same run, same region and season) points to the RCM itself (spin-up of
the land surface); a trend that EC-Earth3 also has is a property of the stabilised climate.

In [ ]:
t = TR[(TR["period"] == TAG) & (TR["model"] != "gcm") & (TR["p"] < P_SIG)]
g = TR[(TR["period"] == TAG) & (TR["model"] == "gcm")].set_index(["var", "season", "region", "run"])
SIG = t.assign(gcm_slope=[g.loc[(a, b, c, d), "slope_per_decade"] for a, b, c, d in zip(t["var"], t.season, t.region, t.run)],
               gcm_p=[g.loc[(a, b, c, d), "p"] for a, b, c, d in zip(t["var"], t.season, t.region, t.run)])
SIG = SIG[["var", "season", "region", "model", "run", "slope_per_decade", "p", "gcm_slope", "gcm_p", "unit"]]
SIG.to_csv(OUT / f"trends_significant_rcm_{TAG}.csv", index=False)
n_all = len(TR[(TR["period"] == TAG) & (TR["model"] != "gcm")])
print(f"RCM series with a trend at p < {P_SIG} over {TAG}: {len(SIG)} of {n_all} "
      f"(about {P_SIG * n_all:.0f} expected by chance; the tests are not independent)")
with pd.option_context("display.float_format", "{:+.2f}".format, "display.max_rows", 200, "display.width", 200):
    for run in RUNS:
        x = SIG[SIG["run"] == run]
        print(f"\n{RUNS[run]}: {len(x)} significant RCM trends")
        if len(x):
            display(x.sort_values(["var", "model", "season", "region"]).reset_index(drop=True))

## 5. Hosing effect: mean over `YEARS` (notebooks 1-3), last decade, and end of the linear fits

In [ ]:
def fit_at(yrs, x, year):
    b, a = np.polyfit(yrs, x, 1)
    return a + b * year


rows = []
for (var, s, m), (yrs, xn, xh) in SER.items():
    w = (yrs >= YEARS[0]) & (yrs <= YEARS[1]); late = (yrs >= LATE[0]) & (yrs <= LATE[1])
    y = yrs[w].astype(float)
    for i, r in enumerate(REGIONS):
        d_tr = trend(y, change(xh[w, i], xn[w, i], var))
        rows.append(dict(var=var, season=s, region=r, model=m, unit="%" if var == "pr" else "K",
                         effect_years=change(xh[w, i].mean(), xn[w, i].mean(), var),
                         effect_late=change(xh[late, i].mean(), xn[late, i].mean(), var),
                         effect_end=change(fit_at(y, xh[w, i], y[-1]), fit_at(y, xn[w, i], y[-1]), var),
                         diff_trend_per_decade=d_tr[0], diff_trend_p=d_tr[2]))
EFF = pd.DataFrame(rows)
EFF["end_minus_years"] = EFF["effect_end"] - EFF["effect_years"]
EFF.to_csv(OUT / f"hosing_effect_drift_{TAG}.csv", index=False)

for var in VARS:
    u = "%" if var == "pr" else "K"
    t = EFF[EFF["var"] == var]
    for m in MODELS:
        x = t[t["model"] == m]
        show = x.assign(v=[f"{a:+.2f} / {b:+.2f} / {c:+.2f}{star(p)}" for a, b, c, p in
                           zip(x.effect_years, x.effect_late, x.effect_end, x.diff_trend_p)])
        print(f"\n{var} ({u}), {MODELS[m]}: hosing effect {TAG} / {LATE[0]}-{LATE[1]} / end of the fits "
              f"(* trend of the difference at p < {P_SIG})")
        display(show.pivot(index="region", columns="season", values="v").loc[REGIONS, list(SEASONS)])

big = EFF[(EFF["diff_trend_p"] < P_SIG)].assign(abs_shift=lambda d: d["end_minus_years"].abs()).sort_values("abs_shift", ascending=False)
print(f"\nseries whose hosing - no hosing difference has a trend at p < {P_SIG}: {len(big)} of {len(EFF)}; largest shifts:")
with pd.option_context("display.float_format", "{:+.2f}".format, "display.width", 200):
    display(big.drop(columns="abs_shift").head(25).reset_index(drop=True))

## 6. Figures: domain-mean yearly series and their trends

In [ ]:
COL = {"nohos": "C0", "hos": "C3"}
for kind in ("runs", "diff"):
    fig, axes = plt.subplots(len(MODELS), 4, figsize=(20, 11), sharex=True)
    for i, m in enumerate(MODELS):
        for j, (var, s) in enumerate((("tas", "year"), ("tas", "SON"), ("pr", "year"), ("pr", "SON"))):
            ax = axes[i, j]
            yrs, xn, xh = SER[var, s, m]
            series = {"nohos": xn[:, 0], "hos": xh[:, 0]} if kind == "runs" else {"diff": change(xh[:, 0], xn[:, 0], var)}
            for run, x in series.items():
                k = (yrs >= YEARS[0]) & (yrs <= YEARS[1])
                b, a = np.polyfit(yrs[k], x[k], 1)
                sl, _, p, _ = trend(yrs[k].astype(float), x[k])
                ax.plot(yrs, x, "o-", ms=3, lw=0.8, color=COL.get(run, "k"), label=f"{RUNS[run]}: {sl:+.2f}/decade (p {p:.2f})")
                ax.plot(yrs[k], a + b * yrs[k], "--", color=COL.get(run, "k"), lw=1.2)
            ax.axvspan(DATA_YEARS[0] - 0.5, YEARS[0] - 0.5, color="0.9", zorder=0)
            ax.set_title(f"{MODELS[m]}: {var} {s}, domain mean ({'mm/day' if var == 'pr' and kind == 'runs' else '%' if var == 'pr' else 'deg C' if kind == 'runs' else 'K'})", fontsize=9)
            ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(OUT / f"trends_{kind}_{DATA_YEARS[0]}-{DATA_YEARS[1]}.png", dpi=110); plt.show()

## 7. Reading

- **Section 3.** ECOADAPT's -0.23 K per decade is the annual land mean of TSA over EUR-075 for stab-2050: compare with
  RegCM5 0.75 deg, tas, year, no hosing, 2125-2149. A trend of the RCM that EC-Earth3 does not have (section 4) is an
  artefact of the RCM; one that EC-Earth3 also has is climate.
- **Section 5.** If the hosing effect over `YEARS`, over the last decade and at the end of the fits agree, the drift does
  not matter for notebooks 1-3. If they differ where the difference hosing - no hosing has a significant trend, the
  effect of notebooks 1-3 contains part of the drift: the end of the fits is the better estimate if the drift is a
  spin-up artefact, but a linear drift that has not levelled off by 2149 leaves the true level unknown.
- **Autumn cooling of RegCM5 0.75 deg** (-1.62 K against -0.64 K for EC-Earth3 in notebook 3): look at tasmax SON in
  section 5 for r075 against gcm.